In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

PROJECT_DIR = Path(os.environ.get("NEISS_PROJECT_DIR", Path.cwd())).expanduser().resolve()
DATA_DIR = Path(
    os.environ.get("NEISS_DATA_DIR", PROJECT_DIR / "data" / "neiss")
).expanduser().resolve()
OUTPUT_DIR = Path(
    os.environ.get("NEISS_OUTPUT_DIR", PROJECT_DIR / "outputs" / "neiss_world_cup_v2")
).expanduser().resolve()
MASTER_DIR = Path(
    os.environ.get("NEISS_MASTER_RESULTS_DIR", PROJECT_DIR / "outputs" / "master_results")
).expanduser().resolve()
TABLE_DIR = OUTPUT_DIR / "tables"
AUDIT_DIR = OUTPUT_DIR / "audit"
MASTER_PATH = MASTER_DIR / "master_results_dictionary.csv"

RUN_PRIMARY_PIPELINE = True
RUN_MANUAL_REVIEW_PASS = True
RUN_PUBLICATION_FIGURES = False

os.environ["NEISS_PROJECT_DIR"] = str(PROJECT_DIR)
os.environ["NEISS_DATA_DIR"] = str(DATA_DIR)
os.environ["NEISS_OUTPUT_DIR"] = str(OUTPUT_DIR)
os.environ["NEISS_MASTER_RESULTS_DIR"] = str(MASTER_DIR)

required_script = PROJECT_DIR / "analysis_v2" / "neiss_world_cup_analysis_v2.py"
required_workbooks = [DATA_DIR / f"neiss{year}.xlsx" for year in range(1999, 2026)]
missing_workbooks = [path.name for path in required_workbooks if not path.exists()]

if not required_script.exists():
    raise FileNotFoundError(
        "The analysis script was not found. Start Jupyter from the reproducibility-package root "
        "or set NEISS_PROJECT_DIR."
    )
if missing_workbooks:
    raise FileNotFoundError(
        f"Missing {len(missing_workbooks)} annual NEISS workbook(s), including "
        f"{missing_workbooks[:5]}. Populate data/neiss or set NEISS_DATA_DIR."
    )

print("Project directory:", PROJECT_DIR)
print("Data directory:", DATA_DIR)
print("Output directory:", OUTPUT_DIR)
print("Annual workbooks found:", len(required_workbooks))
print("Primary denominator wording:")
print("NEISS-estimated emergency department-treated consumer product and recreation-related injuries")


## 2. Run the primary analysis pipeline

In [ ]:
if RUN_PRIMARY_PIPELINE:
    run = subprocess.run(
        [sys.executable, str(PROJECT_DIR / "analysis_v2" / "neiss_world_cup_analysis_v2.py")],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
        env=os.environ.copy(),
    )
    print("Primary pipeline return code:", run.returncode)
    print("\nPipeline stdout tail:")
    print("\n".join(run.stdout.splitlines()[-35:]))
    if run.stderr.strip():
        print("\nPipeline stderr tail:")
        print("\n".join(run.stderr.splitlines()[-35:]))
    assert run.returncode == 0, "The primary analysis pipeline did not complete successfully."
else:
    print("Primary pipeline skipped by configuration.")


## 3. Run the manual-review resolution pass

In [ ]:
if RUN_MANUAL_REVIEW_PASS:
    review_run = subprocess.run(
        [sys.executable, str(PROJECT_DIR / "analysis_v2" / "manual_review_resolution_pass.py")],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
        env=os.environ.copy(),
    )
    print("Manual-review resolution return code:", review_run.returncode)
    print("\n".join(review_run.stdout.splitlines()[-25:]))
    if review_run.stderr.strip():
        print("\nManual-review stderr tail:")
        print("\n".join(review_run.stderr.splitlines()[-25:]))
    assert review_run.returncode == 0, "The manual-review resolution pass did not complete successfully."
else:
    print("Manual-review resolution pass skipped by configuration.")


## 4. Optionally generate publication-quality figure formats

In [ ]:
if RUN_PUBLICATION_FIGURES:
    figure_run = subprocess.run(
        [sys.executable, str(PROJECT_DIR / "analysis_v2" / "generate_publication_figures.py")],
        cwd=PROJECT_DIR,
        capture_output=True,
        text=True,
        env=os.environ.copy(),
    )
    print("Publication-figure return code:", figure_run.returncode)
    print("\n".join(figure_run.stdout.splitlines()[-20:]))
    if figure_run.stderr.strip():
        print("\nPublication-figure stderr tail:")
        print("\n".join(figure_run.stderr.splitlines()[-20:]))
    assert figure_run.returncode == 0, "Publication-quality figure generation did not complete successfully."
else:
    print("Publication-quality figure generation skipped. Set RUN_PUBLICATION_FIGURES = True to run it.")


## Execution Summary

In [ ]:
summary = json.loads((OUTPUT_DIR / "run_summary.json").read_text(encoding="utf-8"))
print(json.dumps(summary, indent=2))


## Cohort and Data Availability

In [ ]:
table_a = pd.read_csv(TABLE_DIR / "table_A_cohort_data_availability.csv")
cols = [
    "year", "database_records", "analysis_eligible_records", "soccer_product_n",
    "weighted_estimate", "ci_lower", "ci_upper", "cv", "stability_flag"
]
print(table_a[cols].to_string(index=False))


## Primary Annual Soccer-Injury Burden

In [ ]:
table_b = pd.read_csv(TABLE_DIR / "table_B_primary_soccer_injury_burden.csv")
print(table_b.to_string(index=False))


## Primary Tournament-Period Comparison

In [ ]:
table_c = pd.read_csv(TABLE_DIR / "table_C_tournament_period_comparison.csv")
print(table_c.to_string(index=False))


## Primary Model Results and Diagnostics

In [ ]:
table_e = pd.read_csv(TABLE_DIR / "table_E_primary_model_results.csv")
primary_terms = table_e[
    table_e["term"].isin(["men_tournament_vs_matched_control", "men_tournament_fraction", "matched_control_fraction"])
]
print(primary_terms.to_string(index=False))
print("\nDiagnostics:")
print(pd.read_csv(TABLE_DIR / "table_E_model_diagnostics.csv").to_string(index=False))


## Sensitivity Analyses

In [ ]:
table_f = pd.read_csv(TABLE_DIR / "table_F_sensitivity_analyses.csv")
print(table_f.to_string(index=False))


## Value Consistency and Output Manifest

In [ ]:
master = pd.read_csv(MASTER_PATH)
manifest = pd.read_csv(OUTPUT_DIR / "output_manifest.csv")
print("Master results entries:", len(master))
print("Unique result IDs:", master["result_id"].nunique())
print("Duplicate result IDs:", int(master["result_id"].duplicated().sum()))
print("Manifest files:", len(manifest))
print("\nMaster results sample:")
print(master.head(25).to_string(index=False))


## Final Figures

![Study flow](outputs/neiss_world_cup_v2/figures/figure1_study_flow.png)

![Annual burden](outputs/neiss_world_cup_v2/figures/figure2_annual_primary_burden.png)

![Tournament comparison](outputs/neiss_world_cup_v2/figures/figure3_tournament_matched_comparison.png)

![Injury characteristics](outputs/neiss_world_cup_v2/figures/figure4_injury_characteristics.png)

![Observed and fitted weekly estimates](outputs/neiss_world_cup_v2/figures/figure5_weekly_observed_fitted.png)


## Interpretation Guardrails

- Results describe NEISS-estimated ED-treated soccer-related injury burden and not
  soccer participation injury incidence.
- World Cup indicators are ecological calendar-period variables.
- The v2 variance estimator is an approximate stratified-PSU with-replacement
  implementation and requires independent verification before submission.
- Product-code plausibility review and narrative keyword validation remain manual
  review tasks.
- LDA, TF-IDF, SVD, and t-SNE are not part of the primary v2 workflow.
